# Memory & Architecture
In this second part of Phase 26, we explore how Python manages memory, organizes its data types through a unified object model, and resolves modules via its internal import system.

## 1. Memory Allocator
CPython manages memory using a two-tier architecture designed to minimize operating system overhead:

The OS / Raw Memory Allocator: Interfaces directly with the operating system (malloc and free) to request large chunks of virtual memory.

The Arena / Pool / Block Allocator (PyMalloc): For small objects (under 512 bytes), Python bypasses the operating system's malloc and uses its own specialized allocator (PyMalloc). It groups memory into Arenas (256 KB), breaks them into Pools (4 KB), and divides those into small Blocks.

Object-Specific Allocators: Used for heavy internal containers like lists, dicts, and tuples to optimize speed and reuse memory chunks efficiently.

## 2. The Object Model
In Python, "everything is an object." This means every piece of data—from an integer literal like 42 to a function or class definition—is represented in C memory by a struct called a PyObject.

Every PyObject contains two essential fields:

ob_refcnt (Reference Count): Tracks how many variables, containers, or pointers currently reference this object. When ob_refcnt drops to zero, the object is immediately destroyed and its memory is reclaimed.

ob_type (Type Pointer): Points to another structure that defines what the object can do (its methods, attributes, and behavior).

### Garbage Collection (Reference Counting + Cyclic GC):
Reference Counting: The primary mechanism. If you assign an object to a variable or put it in a list, its reference count increases; when variables go out of scope or are deleted (del), the count decreases.

The Cyclic Garbage Collector: Reference counting fails when objects reference each other in a closed loop (e.g., Object A points to Object B, and Object B points to Object A), keeping their reference counts above zero indefinitely. Python uses a generational garbage collector (divided into 3 generations: Gen 0, Gen 1, Gen 2) that periodically sweeps through container objects to detect and break these reference cycles.

## 3. The Import System
When you write import math or from my_module import my_function, Python triggers a sophisticated subsystem responsible for locating, loading, and caching modules.

The import process follows three primary steps:

Finder: Searches for the module specification. Python checks sys.modules (a cache of already-imported modules), then looks through built-in modules, and finally scans paths listed in sys.path (which includes the current working directory, PYTHONPATH, and standard library directories).

Loader: Once the module spec is found, the loader reads the source file or compiled bytecode and creates the module object.

sys.modules Cache: The newly loaded module is stored in sys.modules. Any subsequent imports of the same module in your application instantly retrieve it from this cache without hitting the disk.

### Best Practices & Common Pitfalls
Avoid Circular Imports: If Module A imports Module B, and Module B imports Module A at the top level, Python will raise an ImportError or AttributeError because one module isn't fully initialized yet. Fix this by restructuring shared logic into a third module or importing locally inside functions.

Rely on Automatic Memory Management: You rarely need to manually free memory in Python. However, if you are caching millions of objects globally, be aware that they will stay in memory forever unless explicitly deleted or cleared from references.